# NB2 · Classificazione, regolarizzazione e cross-validation

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/battles5/conad-statistical-learning/blob/main/notebooks/NB2_classificazione_regolarizzazione.ipynb)

**logistica su Default, la soglia, ridge e lasso su Hitters, la scelta di lambda**

corso *Apprendimento statistico* per CONAD Nord Ovest · Orso Peruzzi (IFAB)

> come si esegue una cella: clicca dentro e premi **Shift + Invio**; esegui le celle in ordine, dall'alto verso il basso.

due idee del mattino, su due dataset canonici di ISL:

- la **classificazione** con la regressione logistica, sul dataset *Default* (capitolo 4);
- la **regolarizzazione** (ridge e lasso) per domare tanti predittori, sul dataset *Hitters* (capitolo 6);
- e in mezzo la **cross-validation** (capitolo 5) come bussola per scegliere i parametri.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def carica(nome):
    "carica un csv dal repo GitHub (Colab) o dalla cartella locale ../data (fallback)"
    base = "https://raw.githubusercontent.com/battles5/conad-statistical-learning/main/data/"
    try:
        return pd.read_csv(base + nome)
    except Exception:
        return pd.read_csv("../data/" + nome)

## 1. dati: chi sara' insolvente? (Default)

dataset **Default** di ISL: per 10.000 clienti, se sono andati in insolvenza (`default`), se sono studenti (`student`), il saldo della carta (`balance`) e il reddito (`income`). gli importi sono in dollari (dato originale ISL).

In [ ]:
df = carica("Default.csv")
df["default"] = (df["default"] == "Yes").astype(int)
df["student"] = (df["student"] == "Yes").astype(int)
print("tasso di insolvenza:", round(df["default"].mean(), 3))
df.head()

## 2. regressione logistica

prevediamo la probabilita' di insolvenza da saldo, reddito e stato di studente. standardizziamo le variabili dentro una pipeline.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

X = df[["balance", "income", "student"]].values
y = df["default"].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=0, stratify=y)

logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
prob = logit.predict_proba(Xte)[:, 1]
print(classification_report(yte, (prob > 0.5).astype(int),
                            target_names=["non insolvente", "insolvente"], zero_division=0))
print("AUC sul test:", round(roc_auc_score(yte, prob), 3))

## 3. la soglia e' una scelta, non un dato

di default si classifica "insolvente" se la probabilita' supera 0,5. ma la soglia si puo' spostare: abbassarla individua piu' insolventi (piu' veri positivi) al prezzo di piu' falsi allarmi.

> **manopola**: cambia `SOGLIA` (prova 0.2, 0.3, 0.5) e guarda come cambia la matrice di confusione.

In [ ]:
# >>> MANOPOLA: soglia di decisione <<<
SOGLIA = 0.5

pred = (prob > SOGLIA).astype(int)
cm = confusion_matrix(yte, pred)
print("matrice di confusione (righe = vero, colonne = previsto):")
print(pd.DataFrame(cm, index=["vero: no", "vero: si"], columns=["prev: no", "prev: si"]))
veri_pos = cm[1, 1]; falsi_neg = cm[1, 0]
print(f"\ninsolventi individuati: {veri_pos} su {veri_pos + falsi_neg}")

la **curva ROC** riassume tutte le soglie in un colpo solo: ogni punto e' una soglia, e l'area sotto la curva (AUC) misura quanto bene il modello separa i due gruppi. e' la figura del capitolo 4 di ISL.

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(yte, prob)
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color="#00ADCF", lw=2.5, label=f"logistica (AUC = {roc_auc_score(yte, prob):.3f})")
plt.plot([0, 1], [0, 1], "--", color="#9aa3b2", label="caso (modello inutile)")
plt.xlabel("tasso di falsi positivi"); plt.ylabel("tasso di veri positivi")
plt.title("Curva ROC (Default)  (cfr. ISL cap. 4)"); plt.legend(); plt.show()

## 4. troppe variabili: ridge e lasso (Hitters)

con pochi predittori la regolarizzazione conta poco. passiamo a **Hitters** di ISL: lo stipendio di giocatori di baseball spiegato da 19 statistiche di carriera. qui penalizzare i coefficienti fa la differenza. prevediamo `Salary` (in migliaia di dollari).

In [ ]:
hit = carica("Hitters.csv").dropna(subset=["Salary"]).reset_index(drop=True)
y = hit["Salary"].values
Xdf = pd.get_dummies(hit.drop(columns=["Salary"]),
                     columns=["League", "Division", "NewLeague"], drop_first=True)
nomi = list(Xdf.columns)
print("giocatori:", len(hit), " predittori:", Xdf.shape[1])

from sklearn.preprocessing import StandardScaler
Xs = StandardScaler().fit_transform(Xdf.astype(float))
Xtr, Xte, ytr, yte = train_test_split(Xs, y, test_size=0.30, random_state=0)

il **ridge** (penalita' L2) restringe tutti i coefficienti verso lo zero: piu' forte e' la penalita', piu' piccoli diventano.

> **manopola**: cambia `ALPHA` (prova 0.1, 1, 10, 100, 1000) e osserva come si riduce la dimensione dei coefficienti.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

# >>> MANOPOLA: forza della regolarizzazione ridge <<<
ALPHA = 1.0

ridge = Ridge(alpha=ALPHA).fit(Xtr, ytr)
print(f"alpha = {ALPHA}")
print(f"somma dei valori assoluti dei coefficienti: {np.abs(ridge.coef_).sum():.1f}")
print(f"coefficiente piu' grande (in modulo):       {np.abs(ridge.coef_).max():.1f}")
print(f"R2 sul test:                                {r2_score(yte, ridge.predict(Xte)):.3f}")

il **lasso** (penalita' L1) fa qualcosa in piu': azzera del tutto i coefficienti meno utili, facendo **selezione delle variabili**.

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=10.0, max_iter=100000).fit(Xtr, ytr)
nz = int((np.abs(lasso.coef_) > 1e-6).sum())
print(f"lasso (alpha = 10): variabili diverse da zero: {nz} su {len(nomi)}")
tenute = [n for n, b in zip(nomi, lasso.coef_) if abs(b) > 1e-6]
print("variabili tenute:", tenute)

## 5. scegliere lambda con la cross-validation

non scegliamo `alpha` a occhio: lo scegliamo con la k-fold cross-validation, che stima l'errore su dati nuovi. `LassoCV` prova tante penalita' e tiene la migliore.

In [ ]:
from sklearn.linear_model import LassoCV

lcv = LassoCV(cv=10, max_iter=100000, random_state=0).fit(Xtr, ytr)
print("alpha scelto dalla cross-validation:", round(lcv.alpha_, 3))
print("R2 sul test:", round(r2_score(yte, lcv.predict(Xte)), 3))
print("variabili selezionate:", int((np.abs(lcv.coef_) > 1e-6).sum()), "su", len(nomi))

---

### cella bonus: il "percorso" del lasso

al crescere della penalita', i coefficienti del lasso si restringono e a uno a uno vanno a zero. e' la figura 6.6 di ISL.

In [ ]:
# BONUS
alphas = np.logspace(-1, 2.6, 40)
percorso = np.array([Lasso(alpha=a, max_iter=100000).fit(Xtr, ytr).coef_ for a in alphas])

plt.figure(figsize=(8, 5))
for j in range(percorso.shape[1]):
    plt.plot(alphas, percorso[:, j], lw=1.3)
plt.xscale("log"); plt.axhline(0, color="grey", lw=1)
plt.xlabel("alpha = lambda (scala log)"); plt.ylabel("coefficiente")
plt.title("Percorso del lasso: i coefficienti si restringono  (cfr. ISL fig. 6.6)")
plt.show()